# Seminar 9: Neural Ranking on Tabular Data. DCN v2 vs CatBoost

## Goals

In this seminar we will:
1. Understand why **CatBoost is a very hard baseline to beat** on tabular recommendation data.
2. See why naive neural networks often fail on **mixed dense + high-cardinality categorical** features.
3. Learn two practical encoding techniques:
   - **Unified embeddings with multi-hash** for sparse categorical features.
   - **Piecewise Linear Encoding (PLE)** for dense numerical features.
4. Implement a compact **DCN v2-style** model for click-through prediction.
5. Compare the neural pipeline with **CatBoost** on the same Criteo benchmark.

---

## Why This Seminar Matters

In many industrial recommendation and advertising systems, the final scoring model is trained on a large tabular feature matrix:
- dense statistics,
- categorical IDs,
- engineered crosses,
- context features,
- user-item interaction signals.

For such data, **CatBoost** is often the first strong baseline. It handles heterogeneous features extremely well and usually works surprisingly well out of the box.

So the central question of this seminar is:

> **Can we build a neural ranking model that is competitive on tabular recommendation features, and what design choices make this possible?**

We will treat click prediction as a **pointwise ranking surrogate**: the model predicts the probability of click, and that score is then used for ranking.

In [ ]:
# !pip install -r requirements.txt


In [ ]:
import typing as tp
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm


---
## Part 1: Dataset and Problem Setup

We use the **Downsampled Criteo Kaggle Dataset**:
https://www.kaggle.com/datasets/dogrose/downsampled-criteo-kaggle-dataset/data

### What is Criteo?

Criteo is a classical large-scale benchmark for **CTR prediction**. Each row corresponds to an ad impression and contains:
- a binary target `label` indicating click / no click,
- **13 numerical features** `I1`-`I13`,
- **26 categorical features** `C1`-`C26`.

### Why this dataset is good for the seminar

It captures the key difficulties of industrial ranking models:
- numerical features are heavy-tailed and require preprocessing,
- categorical features have very different cardinalities,
- some categorical fields are extremely large,
- the target is sparse and realistic for ranking-style problems.

### Our setup

We will:
1. preprocess dense and sparse columns,
2. study categorical cardinalities,
3. motivate memory-efficient sparse encodings,
4. train a neural model,
5. compare it with CatBoost.


In [ ]:
!mkdir -p ./data
!curl -L -o ./data/downsampled-criteo-kaggle-dataset.zip   https://www.kaggle.com/api/v1/datasets/download/dogrose/downsampled-criteo-kaggle-dataset
!unzip -o ./data/downsampled-criteo-kaggle-dataset.zip -d ./data


In [ ]:
class CriteoDatasetUtils:
    INT_COLS = [f'I{i + 1}' for i in range(13)]
    CAT_COLS = [f'C{i + 1}' for i in range(26)]
    LABEL_COL = 'label'

    @classmethod
    def preprocess_dense_features(cls, lf: pl.LazyFrame) -> pl.LazyFrame:
        expressions = []
        for col in cls.INT_COLS:
            expressions.append(
                pl.col(col).fill_null(0).add(1 if col != 'I2' else 4).log()
            )
        return lf.with_columns(expressions)

    @classmethod
    def preprocess_categorical_features(cls, lf: pl.LazyFrame) -> pl.LazyFrame:
        expressions = []
        for col in cls.CAT_COLS:
            expressions.append(
                pl.col(col).fill_null('00000000').str.to_integer(base=16)
            )
        return lf.with_columns(expressions)

    @classmethod
    def read_and_preprocess(cls, path: str) -> pl.DataFrame:
        lf = pl.scan_parquet(path)
        lf = cls.preprocess_categorical_features(lf)
        lf = cls.preprocess_dense_features(lf)
        return lf.collect()


In [ ]:
DATASETS_PATH = './data'
train_df = CriteoDatasetUtils.read_and_preprocess(
    f'{DATASETS_PATH}/criteo_train_6days_downsampled.parquet'
)
test_df = CriteoDatasetUtils.read_and_preprocess(
    f'{DATASETS_PATH}/criteo_test_1day_downsampled.parquet'
)

train_df.head(5)


In [ ]:
assert train_df.null_count().pipe(sum).item() == 0
assert test_df.null_count().pipe(sum).item() == 0

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('CTR train  :', float(train_df['label'].mean()))
print('CTR test   :', float(test_df['label'].mean()))


### A Small but Important Reminder

Although we call this seminar *neural ranking*, the training objective here is **binary click prediction**.

This is standard in recommender systems:
- train a pointwise model on clicks / non-clicks,
- then use the predicted score to rank candidates.

So mathematically we solve **binary classification**, but operationally we use the model for **ranking**.


---
## Part 2: Why Naive Neural Modeling Is Difficult

Before introducing any architecture, let us first understand the problem.

### Intuition

If we build a plain neural network for tabular ranking, two issues appear immediately:

1. **Dense features are not nicely distributed.**
   Raw counts are heavy-tailed, so the model sees very uneven scales.

2. **Categorical features may have huge vocabularies.**
   The standard solution is an embedding table, but very large vocabularies quickly become expensive.

3. **Feature interactions matter a lot.**
   In ranking, the effect of one feature often depends on another. A plain MLP can learn such interactions, but not always efficiently.

Let us first inspect the categorical cardinalities and estimate the cost of naive embeddings.


In [ ]:
unique_counts = {col: train_df[col].n_unique() for col in CriteoDatasetUtils.CAT_COLS}
sorted_unique_counts = dict(
    sorted(unique_counts.items(), key=lambda item: item[1], reverse=True)
)
sorted_unique_counts


In [ ]:
uniq_ids = sum(sorted_unique_counts.values())
embedding_dim = 256
bytes_in_float = 4
optimizer_multiplier = 4  # params + grads + Adam states
bytes_in_gb = 1024 ** 3

memory_gb = uniq_ids * embedding_dim * bytes_in_float * optimizer_multiplier / bytes_in_gb
print(f'Approximate memory need for naive embeddings: {memory_gb:.2f} GB')


### Formalization

Suppose each categorical field $t \in \{1, \dots, T\}$ has its own vocabulary $V_t$ and its own embedding table.
If the embedding dimension is $d$, then the total number of embedding parameters is

$$
\sum_{t=1}^{T} |V_t| \cdot d.
$$

This looks harmless on paper, but in industrial systems:
- $|V_t|$ can be in the millions,
- optimizer states multiply memory consumption,
- multiple features may each require large embedding tables.

So we need a more memory-efficient idea.


---
## Part 3: Unified Embeddings with Multi-Hash

### Intuition First

The standard embedding approach allocates a separate row for every possible categorical value.
That is simple, but memory-hungry.

A more scalable idea is:
- use **one shared embedding table** of fixed size,
- map categorical values into this table using **hashing**,
- let different features share this global table.

This creates **collisions**: two different values may land in the same embedding row.
But not all collisions are equally bad.

### Key intuition about collisions

- **Intra-feature collisions**: two values from the same categorical field collide.
  This is harmful, because the model has less chance to distinguish them.

- **Inter-feature collisions**: values from different fields collide.
  This can be less harmful, because downstream model parameters still know which field the value came from.

### Why multi-hash helps

Instead of representing each categorical value by a single hash, we can use several hashes.
Then the representation becomes more robust: it is much less likely that two values collide in exactly the same way across all hashes.


### Formal Setup

Let $x = [v_1, v_2, \dots, v_T]$ be the categorical part of one sample, where $v_t \in V_t$.
We use:
- a shared embedding table $\mathbf{E} \in \mathbb{R}^{M \times d}$,
- a hash function $h_t(v)$ mapping a value into $\{0, 1, \dots, M-1\}$,
- a model $f(g(x; \mathbf{E}); \theta)$ on top of the resulting embedding representation.

The training objective is

$$
\arg \min_{\mathbf{E}, \theta} \; \mathbb{L}_D(\mathbf{E}, \theta)
= \arg \min_{\mathbf{E}, \theta} \sum_{(x,y) \in D} \ell\big(f(g(x; \mathbf{E}); \theta), y\big).
$$

### Visual intuition

<div style="width:90%; margin: auto;">

![](https://i.ibb.co/GKMKcvm/unified-embeddings.png)

</div>

The main engineering question is: **how much damage do collisions do?**


### Why Shared Hashing Can Still Work

For two categorical fields, one can decompose the gradient into three parts:
- the ideal collision-free component,
- the bias caused by collisions **within the same field**,
- the bias caused by collisions **across different fields**.

A useful takeaway from the unified embeddings literature is the following:

- intra-feature collisions are the most problematic,
- inter-feature collisions can be partially mitigated by the downstream linear layer,
- therefore it is especially important to reduce **same-field collision damage**,
- and one practical way to do so is **multiple hashes per feature**.

We keep the implementation practical below, but the conceptual message is important:

> **Shared embeddings are viable not because collisions disappear, but because some collisions are more tolerable than others.**

If we keep the formal picture from the paper, then the gradient can be viewed as

$$
\nabla_{E_{h(u)}} \mathbb{L}_D(\mathbf{E}, \theta)
= \text{collisionless term} + \text{intra-feature bias} + \text{inter-feature bias}.
$$

This decomposition is useful conceptually:
- the **collisionless term** is the gradient we would like to have,
- the **intra-feature bias** is the most harmful because it mixes values inside the same field,
- the **inter-feature bias** can be partially disentangled by downstream field-specific parameters.


In [ ]:
class MultihashTransform:
    """Apply multiple hashes to each categorical feature."""

    def __init__(self, cardinality, seeds=None, name='sparse'):
        assert seeds is not None
        self._cardinality = cardinality
        self._name = name
        self._seeds = torch.tensor(seeds)

    def __call__(self, sample: dict[str, tp.Any]) -> dict[str, tp.Any]:
        sample[self._name] = (
            (sample[self._name].unsqueeze(1) + self._seeds) % self._cardinality
        ).long().reshape(-1)
        return sample


In [ ]:
seeds = [[2342 + 13 * i, 7777 + 17 * i] for i in range(26)]
transform = MultihashTransform(10, seeds)
example = {
    'label': torch.tensor(1),
    'dense': torch.randn(13),
    'sparse': torch.arange(26),
}
output = transform(example)
print(output['sparse'])
print('Shape:', output['sparse'].shape)


In [ ]:
class UnifiedEmbeddings(nn.Module):
    def __init__(self, cardinality, embedding_dim):
        super().__init__()
        self.embeddings = nn.Embedding(
            num_embeddings=cardinality,
            embedding_dim=embedding_dim,
        )

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings(ids)


---
## Part 4: Piecewise Linear Encoding for Numerical Features

### Intuition First

Passing raw numerical features directly into an MLP is often suboptimal.
Even after a log transform, the model still has to learn a useful local geometry of the input space.

A natural idea is to discretize a feature into bins. But hard binning loses too much information:
nearby values inside the same bin become identical.

**Piecewise Linear Encoding (PLE)** keeps the bin structure but also preserves the relative position of the value inside the bin.
So it is more informative than hard one-hot binning and more structured than raw scalar input.


### Formal Definition

For one numerical feature, suppose we have bin boundaries
$ b_0 < b_1 < \dots < b_T $.
Then the Piecewise Linear Encoding is a vector

$$
\text{PLE}(x) = [e_1, \dots, e_T] \in \mathbb{R}^T,
$$

where

$$
e_t =
\begin{cases}
0, & x < b_{t-1} \text{ and } t > 1, \\
1, & x \ge b_t \text{ and } t < T, \\
\frac{x - b_{t-1}}{b_t - b_{t-1}}, & \text{otherwise.}
\end{cases}
$$

<div style="width:90%; margin: auto;">

![](https://i.ibb.co/XZtk6fSN/picewise-linear.png)

</div>

In practice, we usually build bins from **quantiles** of the training distribution.
This adapts the representation to the empirical scale of the feature.


In [ ]:
class PiecewiseLinearEncodingTransform:
    """Apply piecewise-linear encoding to dense features."""

    @staticmethod
    def compute_bins(X: torch.Tensor, n_bins: int) -> list[torch.Tensor]:
        return [
            q.unique()
            for q in torch.quantile(
                X, torch.linspace(0.0, 1.0, n_bins + 1).to(X), dim=0
            ).T
        ]

    def __init__(self, dense_train_df, n_bins=32, name='dense'):
        self._name = name
        self._bins = PiecewiseLinearEncodingTransform.compute_bins(
            dense_train_df.to_torch(), n_bins
        )
        self._n_bins = [len(x) - 1 for x in self._bins]
        max_n_bins = max(self._n_bins)
        n_features = len(self._bins)

        self.weight = torch.zeros(n_features, max_n_bins)
        self.bias = torch.zeros(n_features, max_n_bins)

        for i, bin_edges in enumerate(self._bins):
            bin_width = bin_edges.diff()
            w = 1.0 / bin_width
            b = -bin_edges[:-1] / bin_width
            self.weight[i, -1] = w[-1]
            self.bias[i, -1] = b[-1]
            self.weight[i, :self._n_bins[i] - 1] = w[:-1]
            self.bias[i, :self._n_bins[i] - 1] = b[:-1]

    @property
    def n_bins(self):
        return self._n_bins

    def __call__(self, sample: dict[str, tp.Any]) -> dict[str, tp.Any]:
        x = sample[self._name].to(torch.float32).unsqueeze(0)
        x = torch.addcmul(self.bias, self.weight, x[..., None])
        x = torch.cat(
            [
                x[..., :1].clamp_max(1.0),
                x[..., 1:-1].clamp(0.0, 1.0),
                x[..., -1:].clamp_min(0.0),
            ],
            dim=-1,
        )
        sample[self._name] = x.flatten(-2).squeeze(0)
        return sample


In [ ]:
transform = PiecewiseLinearEncodingTransform(
    train_df[CriteoDatasetUtils.INT_COLS],
    name='dense',
)
example = {
    'label': torch.tensor(1),
    'dense': torch.randn(13),
    'sparse': torch.arange(26),
}
output = transform(example)
print('Encoded dense shape:', output['dense'].shape)
print('Bins per feature:', transform.n_bins)


In [ ]:
class PiecewiseLinearEncoding(nn.Identity):
    """Identity layer: encoding is already done in the dataset transform."""
    pass


---
## Part 5: DCN v2

### Intuition First

Why not just feed the encoded features into a large MLP?

Because recommendation quality often depends on **explicit feature interactions**:
- one user feature matters only for a certain item family,
- one context feature changes the meaning of an item feature,
- click probability may depend on low-order crosses that are important but hard for an MLP to learn efficiently.

The idea of **Deep & Cross Networks** is to explicitly model such interactions.

### Cross layer intuition

A cross layer takes the original input $x_0$ and the current hidden vector $x_l$ and builds feature interactions of the form

$$
x_{l+1} = x_0 \odot (W_l x_l + b_l) + x_l.
$$

So each layer keeps the current representation and adds multiplicative interactions anchored to the original input.
This is why DCN is often effective on tabular data.

<div style="width:50%; margin: auto;">

![](https://i.ibb.co/ZqfF5yf/dcn-v2.png)
![](https://i.ibb.co/SDYWNSMy/dcn-v2-equation.png)

</div>


### Architecture Used in This Seminar

We use a stacked DCN-style pipeline:
1. Encode categorical features with **unified embeddings**.
2. Encode numerical features with **PLE**.
3. Concatenate dense and sparse representations.
4. Pass the result through several **cross layers**.
5. Refine with a **deep network**.
6. Produce one scalar logit for click prediction.

The architectural roles are:
- **Unified embeddings**: memory-efficient sparse representation.
- **PLE**: better geometry for dense inputs.
- **Cross network**: explicit low-order feature interactions.
- **Deep network**: nonlinear refinement on top of crossed features.

<div style="width:50%; margin: auto;">

![](https://i.ibb.co/K34HqJH/dcn-theory.png)

</div>


In [ ]:
class CrossLayer(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, input_dim)

    def forward(self, x0, xl):
        return x0 * self.linear(xl) + xl


class CrossNetwork(nn.Module):
    def __init__(self, input_dim, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([CrossLayer(input_dim) for _ in range(num_layers)])

    def forward(self, x):
        xl = x
        for layer in self.layers:
            xl = layer(x, xl)
        return xl


class DeepNetwork(nn.Module):
    def __init__(self, input_dim, hidden_units):
        super().__init__()
        layers = []
        for units in hidden_units:
            layers.append(nn.Linear(input_dim, units))
            layers.append(nn.ReLU())
            input_dim = units
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class DCNV2(nn.Module):
    def __init__(self, embedding_size, cross_layers, deep_units, input_size, cardinality=65536):
        super().__init__()
        self.sparse_encode_layer = UnifiedEmbeddings(cardinality, embedding_size)
        self.dense_encode_layer = PiecewiseLinearEncoding()
        self.cross_network = CrossNetwork(input_size, cross_layers)
        self.deep_network = DeepNetwork(input_size, deep_units)
        self.output_layer = nn.Linear(deep_units[-1], 1)

    def forward(self, dense_input, sparse_input):
        sparse_embeddings = self.sparse_encode_layer(sparse_input).view(sparse_input.size(0), -1)
        dense_embeddings = self.dense_encode_layer(dense_input)
        combined_input = torch.cat([dense_embeddings, sparse_embeddings], dim=-1)
        cross_output = self.cross_network(combined_input)
        deep_output = self.deep_network(cross_output)
        return self.output_layer(deep_output).squeeze(dim=-1)


---
## Part 6: Training the Neural Ranking Model

We now assemble the full pipeline.

### Training recipe

- target: binary click label,
- loss: `BCEWithLogitsLoss`,
- optimizer: Adam,
- model score: a **logit**,
- primary metric: **ROC-AUC**.

### Helpful practical note

Since the model outputs **logits**, the correct decision threshold for a 0.5 probability is:

$$
\sigma(z) > 0.5 \quad \Longleftrightarrow \quad z > 0.
$$

So if we compute accuracy from logits directly, we should threshold at **0.0**, not at **0.5**.


In [ ]:
class CriteoDataset(Dataset):
    def __init__(self, df: pl.DataFrame, transforms: list[tp.Callable[[tp.Any], tp.Any]] | None = None):
        self._labels = df[CriteoDatasetUtils.LABEL_COL].to_torch().to(torch.float32)
        self._dense = df[CriteoDatasetUtils.INT_COLS].to_torch()
        self._sparse = df[CriteoDatasetUtils.CAT_COLS].to_torch()
        self._transforms = transforms if transforms is not None else []

    def __len__(self):
        return self._labels.size(0)

    def __getitem__(self, idx):
        sample = {
            'label': self._labels[idx],
            'dense_features': self._dense[idx],
            'sparse_features': self._sparse[idx],
        }
        for transform in self._transforms:
            sample = transform(sample)
        return sample


In [ ]:
batch_size = 4096
cardinality = 8 * 65536
seeds = [
    [2342 + 13 * i, 7777 + 17 * i, 131 + 833 * i]
    for i in range(len(CriteoDatasetUtils.CAT_COLS))
]
num_hashes = 3
embedding_size = 64
n_bins = 39


In [ ]:
dense_transform = PiecewiseLinearEncodingTransform(
    train_df[CriteoDatasetUtils.INT_COLS],
    n_bins=n_bins,
    name='dense_features',
)
sparse_transform = MultihashTransform(
    cardinality=cardinality,
    seeds=seeds,
    name='sparse_features',
)
transforms = [dense_transform, sparse_transform]

train_dataset = CriteoDataset(train_df, transforms)
val_dataset = CriteoDataset(test_df, transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


In [ ]:
def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch in tqdm(train_loader, desc=f'Train epoch {epoch + 1}'):
            dense_features = batch['dense_features'].to(device)
            sparse_features = batch['sparse_features'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            logits = model(dense_features, sparse_features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        total = 0
        correct = 0
        all_logits = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'Val epoch {epoch + 1}'):
                dense_features = batch['dense_features'].to(device)
                sparse_features = batch['sparse_features'].to(device)
                labels = batch['label'].to(device)

                logits = model(dense_features, sparse_features)
                loss = criterion(logits, labels)
                probs = torch.sigmoid(logits)
                preds = (logits > 0.0).float()

                val_loss += loss.item()
                total += labels.size(0)
                correct += (preds == labels).sum().item()
                all_logits.append(logits.cpu())
                all_labels.append(labels.cpu())

        all_logits = torch.cat(all_logits, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        all_probs = torch.sigmoid(all_logits)

        print(
            f'Epoch {epoch + 1}/{epochs} | '
            f'Train Loss: {train_loss / len(train_loader):.4f} | '
            f'Val Loss: {val_loss / len(val_loader):.4f} | '
            f'Accuracy: {100 * correct / total:.2f}% | '
            f'ROC-AUC: {roc_auc_score(all_labels, all_logits):.4f} | '
            f'PR-AUC: {average_precision_score(all_labels, all_probs):.4f}'
        )


In [ ]:
input_size = (
    max(dense_transform.n_bins) * len(CriteoDatasetUtils.INT_COLS)
    + num_hashes * embedding_size * len(CriteoDatasetUtils.CAT_COLS)
)

model = DCNV2(
    embedding_size=embedding_size,
    cross_layers=3,
    deep_units=[1024, 1024, 1024],
    input_size=input_size,
    cardinality=cardinality,
)

print('Input size:', input_size)


In [ ]:
train_model(model, train_loader, val_loader, epochs=1, lr=1e-4)


---
## Part 7: Comparison with CatBoost

### Why compare with CatBoost?

This comparison is pedagogically important.

If CatBoost is stronger, that does **not** mean the neural approach was useless.
It tells us something important:
- CatBoost remains a very strong baseline for tabular ranking,
- neural models need the right feature encoding and enough tuning,
- improvements from architecture alone are rarely automatic.

### Fair comparison principles

We want the comparison to be as fair as possible:
- same train / validation split,
- same supervised target,
- same evaluation metric,
- both models see the same raw columns.

The difference is in how they process features:
- CatBoost handles categorical features internally,
- the neural model uses hashing-based embeddings and PLE.

### A small GPU-specific note

When CatBoost is trained on GPU with `eval_metric='AUC'`, it cannot compute AUC on every iteration directly on GPU.
So CatBoost evaluates this metric only every few iterations and prints a warning-like message such as:
`Default metric period is 5 because AUC is/are not implemented for GPU`.

This is expected behavior, not an error. To make the notebook cleaner, we set `metric_period` explicitly below.

From the DCN v2 paper, the motivation is that explicit cross modeling can bring substantial quality improvements in ranking systems:

<div style="width:50%; margin: auto;">

![](https://i.ibb.co/HDHJ8Nzq/level-improvement.png)
![](https://i.ibb.co/fYpyrKBs/table.png)

</div>


In [ ]:
# GPU support is included in the main CatBoost package.
# !pip install -U catboost

In [ ]:
import gc

# Free GPU memory used by the PyTorch DCN model before CatBoost starts.
# This helps avoid CatBoost warnings about low available GPU memory.
for name in ['model', 'train_loader', 'val_loader', 'train_dataset', 'val_dataset']:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f'GPU memory after cleanup: free={free_bytes / 1024**3:.2f} GB / total={total_bytes / 1024**3:.2f} GB')


In [ ]:
import catboost as cb

X_train = train_df.drop('label').to_pandas()
y_train = train_df['label'].to_pandas()
X_test = test_df.drop('label').to_pandas()
y_test = test_df['label'].to_pandas()

train_pool = cb.Pool(X_train, y_train, cat_features=CriteoDatasetUtils.CAT_COLS)
test_pool = cb.Pool(X_test, y_test, cat_features=CriteoDatasetUtils.CAT_COLS)

catboost_model = cb.CatBoostClassifier(
    iterations=1000,
    loss_function='Logloss',
    eval_metric='AUC',
    early_stopping_rounds=50,
    task_type='GPU',
    metric_period=50,
    verbose=50,
)

In [ ]:
catboost_model.fit(train_pool, eval_set=test_pool, use_best_model=True)

In [ ]:
catboost_probs = catboost_model.predict_proba(X_test)[:, 1]
print('CatBoost ROC-AUC:', roc_auc_score(y_test, catboost_probs))
print('CatBoost PR-AUC :', average_precision_score(y_test, catboost_probs))

---
## Summary

In this seminar we:

1. **Motivated neural ranking on tabular data** by starting from a practical fact: CatBoost is a very strong baseline, and neural models need careful feature design to be competitive.

2. **Studied the structure of the Criteo dataset** and saw why tabular recommendation problems are difficult for neural models: numerical features are skewed, while categorical features can have extremely large cardinalities.

3. **Introduced unified embeddings with multi-hash** as a memory-efficient way to handle high-cardinality categorical inputs. The key theoretical idea is that not all hash collisions are equally harmful: intra-feature collisions are more problematic than inter-feature ones.

4. **Introduced Piecewise Linear Encoding (PLE)** for dense numerical features. This gives a richer representation than raw scalars and preserves local structure better than hard binning.

5. **Implemented a compact DCN v2-style model** that combines:
   - shared sparse embeddings,
   - PLE for dense features,
   - explicit feature crosses,
   - a deep nonlinear backbone.

6. **Compared the neural model with CatBoost** on the same benchmark and discussed the practical lesson: for tabular ranking, success often depends more on input encoding and interaction modeling than on simply using a deeper network.

## Conclusion

A good mental model for this seminar is the following:
- **CatBoost** is still the default baseline for tabular ranking tasks.
- **Neural ranking** becomes attractive when we want scalable sparse representations, shared embedding infrastructure, and architectures that can be extended further.
- The most important design choice is often not the optimizer or the number of layers, but **how we represent dense and sparse features before the ranking model sees them**.
